# Entropy Dataset Preparation for Adaptive Exams

This notebook prepares a leakage-safe dataset for training a classifier that estimates P(correct | question features).

The output is designed for entropy-based question selection in adaptive testing.

Scope: preprocessing only (cleaning, split, feature engineering, encoding, scaling, export).

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TRAIN_RATIO = 0.60
POOL_RATIO = 0.20
TEST_RATIO = 0.20

SMOOTHING_STRENGTH = 20.0
TARGET_COL = "correct"
REQUIRED_COLS = ["student_id", "question_id", "difficulty", "correct", "response_time", "topic"]
DIFFICULTY_MAP = {"easy": 0, "medium": 1, "hard": 2}

DATA_PATH = Path("../Dataset/ready.csv")
OUTPUT_DIR = Path("../Dataset/processed")
MODELS_DIR = Path("../models")

TRAIN_PREPARED_PATH = OUTPUT_DIR / "train_prepared.csv"
POOL_PREPARED_PATH = OUTPUT_DIR / "pool_prepared.csv"
TEST_PREPARED_PATH = OUTPUT_DIR / "test_prepared.csv"

SCALER_PATH = MODELS_DIR / "entropy_scaler.joblib"
QUESTION_AGG_PATH = MODELS_DIR / "entropy_question_aggregates.joblib"
METADATA_PATH = MODELS_DIR / "entropy_metadata.joblib"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

ratio_sum = TRAIN_RATIO + POOL_RATIO + TEST_RATIO
if not np.isclose(ratio_sum, 1.0):
    raise ValueError(f"Split ratios must sum to 1.0, got {ratio_sum}")

In [11]:
df_raw = pd.read_csv(DATA_PATH)

normalized_columns = {col: col.strip().lower().replace(" ", "_") for col in df_raw.columns}
df_raw = df_raw.rename(columns=normalized_columns)

missing_required = [col for col in REQUIRED_COLS if col not in df_raw.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print(f"Loaded {len(df_raw):,} rows and {len(df_raw.columns)} columns from {DATA_PATH}")
print("Columns:", list(df_raw.columns))

df_raw.head()

Loaded 17,340 rows and 6 columns from ..\Dataset\ready.csv
Columns: ['student_id', 'question_id', 'difficulty', 'correct', 'response_time', 'topic']


,student_id,question_id,difficulty,correct,response_time,topic
0,AQklHbi5bt8l,1,hard,0,27232,Misc / Tools
1,AQklHbi5bt8l,2,easy,1,8933,Misc / Tools
2,AQklHbi5bt8l,3,easy,1,14965,Misc / Tools
3,AQklHbi5bt8l,4,easy,1,12691,Misc / Tools
4,AQklHbi5bt8l,5,medium,0,26452,Misc / Tools


In [12]:
df = df_raw.copy()

df["difficulty"] = df["difficulty"].astype(str).str.strip().str.lower()
df["topic"] = df["topic"].astype(str).str.strip()
df["topic"] = df["topic"].replace({"": np.nan, "nan": np.nan, "None": np.nan}).fillna("unknown")

for col in ["question_id", "correct", "response_time"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.drop_duplicates()
df = df.dropna(subset=REQUIRED_COLS).reset_index(drop=True)
df = df[df[TARGET_COL].isin([0, 1])].reset_index(drop=True)

df["question_id"] = df["question_id"].astype(int)
df[TARGET_COL] = df[TARGET_COL].astype(int)
df["response_time"] = df["response_time"].astype(float)

print(f"Cleaned rows: {len(df):,}")
print("Target distribution:")
display(df[TARGET_COL].value_counts(normalize=True).rename("ratio").to_frame())

print("Difficulty distribution:")
display(df["difficulty"].value_counts(normalize=True).rename("ratio").to_frame())

print(f"Unique topics: {df['topic'].nunique()}")
df.head()

Cleaned rows: 17,340
Target distribution:


,ratio
correct,
1,0.757439
0,0.242561


Difficulty distribution:


,ratio
difficulty,
easy,0.725663
medium,0.146828
hard,0.127509


Unique topics: 5


,student_id,question_id,difficulty,correct,response_time,topic
0,AQklHbi5bt8l,1,hard,0,27232.0,Misc / Tools
1,AQklHbi5bt8l,2,easy,1,8933.0,Misc / Tools
2,AQklHbi5bt8l,3,easy,1,14965.0,Misc / Tools
3,AQklHbi5bt8l,4,easy,1,12691.0,Misc / Tools
4,AQklHbi5bt8l,5,medium,0,26452.0,Misc / Tools


In [13]:
stratify_full = df[TARGET_COL] if df[TARGET_COL].nunique() > 1 else None
train_pool_df, test_df = train_test_split(
    df,
    test_size=TEST_RATIO,
    random_state=RANDOM_STATE,
    stratify=stratify_full,
    shuffle=True,
    )

pool_ratio_within_train_pool = POOL_RATIO / (TRAIN_RATIO + POOL_RATIO)
stratify_train_pool = train_pool_df[TARGET_COL] if train_pool_df[TARGET_COL].nunique() > 1 else None
train_df, pool_df = train_test_split(
    train_pool_df,
    test_size=pool_ratio_within_train_pool,
    random_state=RANDOM_STATE,
    stratify=stratify_train_pool,
    shuffle=True,
    )

print(f"Train rows: {len(train_df):,}")
print(f"Pool rows:  {len(pool_df):,}")
print(f"Test rows:  {len(test_df):,}")
print(f"Total check: {len(train_df) + len(pool_df) + len(test_df):,} (expected {len(df):,})")

print("Class ratio by split:")
for name, split_df in [("train", train_df), ("pool", pool_df), ("test", test_df)]:
    ratio = split_df[TARGET_COL].mean()
    print(f"- {name}: {ratio:.4f}")

Train rows: 10,404
Pool rows:  3,468
Test rows:  3,468
Total check: 17,340 (expected 17,340)
Class ratio by split:
- train: 0.7574
- pool: 0.7575
- test: 0.7575


## Train-Only Question Aggregates (Smoothed)

Question statistics are learned only from the training split.

The question correctness rate is smoothed to avoid extreme 0 or 1 values for low-count questions.

In [14]:
def compute_question_aggregates(train_split, smoothing_strength):
    global_correct_rate = float(train_split[TARGET_COL].mean())

    question_stats = train_split.groupby("question_id").agg(
        question_correct_sum=(TARGET_COL, "sum"),
        question_interaction_count=(TARGET_COL, "size"),
        question_avg_response_time=("response_time", "mean"),
        question_response_time_std=("response_time", "std"),
    )

    question_stats["question_correct_rate"] = (
        question_stats["question_correct_sum"] + smoothing_strength * global_correct_rate
    ) / (question_stats["question_interaction_count"] + smoothing_strength)

    question_stats = question_stats.drop(columns=["question_correct_sum"])

    defaults = {
        "question_correct_rate": global_correct_rate,
        "question_avg_response_time": float(train_split["response_time"].mean()),
        "question_response_time_std": float(train_split["response_time"].std(ddof=1)),
        "question_interaction_count": float(question_stats["question_interaction_count"].median()),
    }

    if np.isnan(defaults["question_response_time_std"]):
        defaults["question_response_time_std"] = 0.0

    return question_stats, defaults, global_correct_rate


def apply_question_features(split_df, question_stats, defaults):
    out = split_df.copy()

    for col in [
        "question_correct_rate",
        "question_avg_response_time",
        "question_response_time_std",
        "question_interaction_count",
    ]:
        out[col] = out["question_id"].map(question_stats[col]).fillna(defaults[col])

    out["difficulty_encoded"] = out["difficulty"].map(DIFFICULTY_MAP).fillna(1).astype(int)
    out["response_time_log"] = np.log1p(out["response_time"].clip(lower=0))

    out["topic"] = out["topic"].astype(str).str.strip()
    out.loc[out["topic"].eq(""), "topic"] = "unknown"

    return out


question_stats, question_defaults, global_correct_rate = compute_question_aggregates(
    train_df,
    smoothing_strength=SMOOTHING_STRENGTH,
    )

train_feat = apply_question_features(train_df, question_stats, question_defaults)
pool_feat = apply_question_features(pool_df, question_stats, question_defaults)
test_feat = apply_question_features(test_df, question_stats, question_defaults)

print("Question-level feature engineering complete.")
print(f"Train rows: {len(train_feat):,} | Pool rows: {len(pool_feat):,} | Test rows: {len(test_feat):,}")

Question-level feature engineering complete.
Train rows: 10,404 | Pool rows: 3,468 | Test rows: 3,468


In [15]:
topic_levels = sorted(train_feat["topic"].dropna().unique().tolist())
if "unknown" not in topic_levels:
    topic_levels.append("unknown")


def encode_topic(split_df, known_topic_levels):
    out = split_df.copy()
    out.loc[~out["topic"].isin(known_topic_levels), "topic"] = "unknown"
    out["topic"] = pd.Categorical(out["topic"], categories=known_topic_levels)
    topic_ohe = pd.get_dummies(out["topic"], prefix="topic", dtype=int)
    out = pd.concat([out.drop(columns=["topic"]), topic_ohe], axis=1)
    return out


train_enc = encode_topic(train_feat, topic_levels)
pool_enc = encode_topic(pool_feat, topic_levels)
test_enc = encode_topic(test_feat, topic_levels)

topic_cols = [col for col in train_enc.columns if col.startswith("topic_")]
for split_df in [train_enc, pool_enc, test_enc]:
    for topic_col in topic_cols:
        split_df[f"difficulty_x_{topic_col}"] = split_df["difficulty_encoded"] * split_df[topic_col]

pool_enc = pool_enc.reindex(columns=train_enc.columns, fill_value=0)
test_enc = test_enc.reindex(columns=train_enc.columns, fill_value=0)

exclude_from_features = [
    "student_id",
    "question_id",
    TARGET_COL,
    "difficulty",
    "response_time",
]

feature_cols = [col for col in train_enc.columns if col not in exclude_from_features]

X_train = train_enc[feature_cols].copy()
X_pool = pool_enc[feature_cols].copy()
X_test = test_enc[feature_cols].copy()

y_train = train_enc[TARGET_COL].astype(int).copy()
y_pool = pool_enc[TARGET_COL].astype(int).copy()
y_test = test_enc[TARGET_COL].astype(int).copy()

print(f"Feature count: {len(feature_cols)}")
print(f"X_train: {X_train.shape} | X_pool: {X_pool.shape} | X_test: {X_test.shape}")

Feature count: 18
X_train: (10404, 18) | X_pool: (3468, 18) | X_test: (3468, 18)


In [16]:
numeric_scale_cols = [
    "question_correct_rate",
    "question_avg_response_time",
    "question_response_time_std",
    "question_interaction_count",
    "response_time_log",
]
numeric_scale_cols = [col for col in numeric_scale_cols if col in feature_cols]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_pool_scaled = X_pool.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_scale_cols] = scaler.fit_transform(X_train[numeric_scale_cols])
X_pool_scaled[numeric_scale_cols] = scaler.transform(X_pool[numeric_scale_cols])
X_test_scaled[numeric_scale_cols] = scaler.transform(X_test[numeric_scale_cols])

train_prepared = pd.concat(
    [
        train_enc[["question_id"]].reset_index(drop=True),
        X_train_scaled.reset_index(drop=True),
        y_train.reset_index(drop=True),
    ],
    axis=1,
    )
pool_prepared = pd.concat(
    [
        pool_enc[["question_id"]].reset_index(drop=True),
        X_pool_scaled.reset_index(drop=True),
        y_pool.reset_index(drop=True),
    ],
    axis=1,
    )
test_prepared = pd.concat(
    [
        test_enc[["question_id"]].reset_index(drop=True),
        X_test_scaled.reset_index(drop=True),
        y_test.reset_index(drop=True),
    ],
    axis=1,
    )

print("Scaling complete.")
print(f"Scaled numeric feature count: {len(numeric_scale_cols)}")
print(f"Prepared shapes -> train: {train_prepared.shape}, pool: {pool_prepared.shape}, test: {test_prepared.shape}")

Scaling complete.
Scaled numeric feature count: 5
Prepared shapes -> train: (10404, 20), pool: (3468, 20), test: (3468, 20)


In [17]:
train_prepared.to_csv(TRAIN_PREPARED_PATH, index=False)
pool_prepared.to_csv(POOL_PREPARED_PATH, index=False)
test_prepared.to_csv(TEST_PREPARED_PATH, index=False)

question_aggregates_artifact = {
    "question_stats": question_stats,
    "defaults": question_defaults,
    "global_correct_rate": global_correct_rate,
    "smoothing_strength": SMOOTHING_STRENGTH,
}

metadata = {
    "random_state": RANDOM_STATE,
    "target_col": TARGET_COL,
    "feature_columns": feature_cols,
    "numeric_scaled_columns": numeric_scale_cols,
    "topic_levels": topic_levels,
    "difficulty_map": DIFFICULTY_MAP,
    "split_sizes": {
        "train": int(len(train_df)),
        "pool": int(len(pool_df)),
        "test": int(len(test_df)),
    },
}

dump(scaler, SCALER_PATH)
dump(question_aggregates_artifact, QUESTION_AGG_PATH)
dump(metadata, METADATA_PATH)

print("Saved prepared datasets:")
print(f"- {TRAIN_PREPARED_PATH}")
print(f"- {POOL_PREPARED_PATH}")
print(f"- {TEST_PREPARED_PATH}")

print("Saved artifacts:")
print(f"- {SCALER_PATH}")
print(f"- {QUESTION_AGG_PATH}")
print(f"- {METADATA_PATH}")

Saved prepared datasets:
- ..\Dataset\processed\train_prepared.csv
- ..\Dataset\processed\pool_prepared.csv
- ..\Dataset\processed\test_prepared.csv
Saved artifacts:
- ..\models\entropy_scaler.joblib
- ..\models\entropy_question_aggregates.joblib
- ..\models\entropy_metadata.joblib


In [18]:
prepared_splits = {
    "train": train_prepared,
    "pool": pool_prepared,
    "test": test_prepared,
}

for split_name, split_df in prepared_splits.items():
    forbidden_cols = [col for col in split_df.columns if col == "student_id" or col.startswith("student_")]
    if forbidden_cols:
        raise AssertionError(f"{split_name}: found forbidden columns {forbidden_cols}")

    if "response_time" in split_df.columns:
        raise AssertionError(f"{split_name}: raw response_time should not be present")

    if split_df.isna().any().any():
        nan_cols = split_df.columns[split_df.isna().any()].tolist()
        raise AssertionError(f"{split_name}: NaNs found in {nan_cols}")

model_feature_cols = [col for col in train_prepared.columns if col not in ["question_id", TARGET_COL]]
for split_name, split_df in prepared_splits.items():
    cols = [col for col in split_df.columns if col not in ["question_id", TARGET_COL]]
    if cols != model_feature_cols:
        raise AssertionError(f"{split_name}: feature columns are not aligned with train")

print("Validation checks passed:")
print("- No student_id or student-level features")
print("- Raw response_time removed")
print("- No NaNs in prepared datasets")
print("- Same feature columns across train/pool/test")

Validation checks passed:
- No student_id or student-level features
- Raw response_time removed
- No NaNs in prepared datasets
- Same feature columns across train/pool/test


## Preparation Summary

This notebook now enforces the entropy-preparation constraints:

- question-level train-only aggregates with smoothing
- difficulty x topic interaction features
- scaling only for aggregate metrics and response_time_log
- no student-level features
- no raw response_time in prepared datasets
- aligned train/pool/test feature schema